# 02. Construct Validity: Alignment With the Induction Design (Appendix E.2; Table A2)

> Reproduction notebook for *Addressing the Psychological Small-Sample Problem with Multimodal Large Language Models: Evidence from Multiple Moral Emotion Recognition Study*. Section numbers refer to the manuscript appendices. Every notebook reads only the committed files in `../final/` and ends with a check against the frozen outputs in `../final/reproducibility/`. Run the notebooks in numerical order; `00` writes the label tables that the others reuse.

Implements Appendix E.2 for the ten MLLM configurations (Exp 1–9 and the supplementary control Exp 16; Table A2). The four induction cells are treated as known groups. For each MLLM configuration a 2 × 2 between-participant ANOVA (Type II sums of squares) relates each continuous score (Sympathy, Inspiration) to the two manipulated factors. Partial η² = SS_effect / (SS_effect + SS_error) is reported with a 90% interval obtained by inverting the noncentral *F* distribution. Construct validity predicts a diagonal structure: each manipulation should have its strongest association with the matching score.

The notebook also computes the threshold-free one-vs-rest macro-AUC of the supplementary discretization-sensitivity analysis (notebook 06; not reported in the manuscript), from the two continuous scores by projecting the centred score pair onto the four class directions (Elevation +S +I, Sympathy +S −I, Inspiration −S +I, Control −S −I); notebook 06 recomputes it and verifies it against the frozen file.

In [1]:
import os, sys, json
import numpy as np, pandas as pd
sys.path.insert(0, os.path.abspath("."))
import common as C
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 40)
rng = np.random.default_rng(C.SEED)
print("final data folder:", os.path.relpath(C.FINAL))

final data folder: ../final


## 1. Load labels and scores

In [2]:
ALL = pd.read_csv(os.path.join(C.OUT, "labels_all.csv"))
LAB = {e: g.drop(columns="exp").reset_index(drop=True) for e, g in ALL.groupby("exp") if e in C.MLLM_EXPS}

## 2. Type II 2 × 2 ANOVAs with partial η² and 90% intervals

In [3]:
AN = pd.concat([C.anova_2x2(LAB[e], e) for e in C.MLLM_EXPS], ignore_index=True)
AUC = {e: C.macro_auc(LAB[e]) for e in C.MLLM_EXPS}; AN["macro_auc"] = AN.exp.map(AUC)
AN.to_csv(os.path.join(C.OUT, "anova.csv"), index=False)
tab = AN[AN.effect != "Interaction"].pivot_table(index="exp", columns=["score", "effect"], values="eta_p2").round(2)
tab["macro_AUC"] = pd.Series(AUC).round(2); tab

Out[3]: 
score               insp                             symp                macro_AUC
effect Inspiration manip Sympathy manip Inspiration manip Sympathy manip          
exp                                                                               
1                   0.03           0.03              0.01           0.01      0.57
2                   0.07           0.04              0.01           0.23      0.70
3                   0.41           0.03              0.00           0.38      0.82
4                   0.51           0.01              0.07           0.59      0.90
5                   0.45           0.05              0.00           0.46      0.84
6                   0.51           0.01              0.00           0.52      0.87
7                   0.44           0.11              0.00           0.32      0.80
8                   0.46           0.00              0.00           0.39      0.84
9                   0.54           0.02              0.01           0.67      

## 3. Exceptions to the design-alignment pattern (Table A2)

This section lists the incongruent effects (a manipulation acting on the non-matching score) and the interactions that reach *p* < .05; Table A2 gives the largest incongruent effect and the interaction terms per configuration.

In [4]:
I = AN[AN.effect == "Interaction"]; print("interactions with p < .05:"); print(I[I.p < .05][["exp", "score", "F", "p", "eta_p2"]].round(3).to_string(index=False))
inc = AN[((AN.score == "symp") & (AN.effect == "Inspiration manip")) | ((AN.score == "insp") & (AN.effect == "Sympathy manip"))]
print("\nincongruent effects with p < .05:"); print(inc[inc.p < .05][["exp", "score", "effect", "p", "eta_p2"]].round(3).to_string(index=False))
print("\nlargest incongruent partial eta^2 per configuration:"); print(inc.groupby("exp").eta_p2.max().round(3).to_string())

interactions with p < .05:
 exp score     F     p  eta_p2
   3  insp 6.194 0.015   0.061
   5  symp 6.828 0.010   0.066
   6  symp 5.370 0.023   0.053
   7  symp 7.497 0.007   0.072
   7  insp 7.412 0.008   0.072
   8  symp 4.030 0.048   0.040

incongruent effects with p < .05:
 exp score            effect     p  eta_p2
   2  insp    Sympathy manip 0.047   0.041
   4  symp Inspiration manip 0.008   0.072
   5  insp    Sympathy manip 0.033   0.046
   7  insp    Sympathy manip 0.001   0.112
  16  symp Inspiration manip 0.046   0.041

largest incongruent partial eta^2 per configuration:
exp
1     0.029
2     0.041
3     0.027
4     0.072
5     0.046
6     0.012
7     0.112
8     0.002
9     0.021
16    0.041


## 4. Verification against the frozen outputs

In [5]:
fz = C.load_frozen("anova_frozen.csv").replace({"Admiration manip": "Inspiration manip"})
print(C.compare(AN, fz, ["exp", "score", "effect"], ["N", "df1", "df2", "F", "p", "eta_p2", "eta_lo90", "eta_hi90", "macro_auc"], atol=0.001).to_string(index=False))

   column  n_compared  max_abs_diff  within_tolerance
        N          60  0.000000e+00              True
      df1          60  0.000000e+00              True
      df2          60  0.000000e+00              True
        F          60  4.547474e-13              True
        p          60  5.884182e-15              True
   eta_p2          60  7.216450e-16              True
 eta_lo90          60  1.387779e-15              True
 eta_hi90          60  1.443290e-15              True
macro_auc          60  1.110223e-16              True
